In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

In [2]:
customers = pd.read_csv(r"C:\Users\sanju\OneDrive\Attachments\olist_customers_dataset.csv")
orders = pd.read_csv(r"C:\Users\sanju\OneDrive\Attachments\olist_orders_dataset.csv")
order_items = pd.read_csv(r"C:\Users\sanju\OneDrive\Attachments\olist_order_items_dataset.csv")

payments = pd.read_csv(r"C:\Users\sanju\OneDrive\Attachments\olist_order_payments_dataset.csv")

reviews = pd.read_csv(r"C:\Users\sanju\OneDrive\Attachments\olist_order_reviews_dataset.csv")

products = pd.read_csv(r"C:\Users\sanju\OneDrive\Attachments\olist_products_dataset.csv")
sellers = pd.read_csv(r"C:\Users\sanju\OneDrive\Attachments\olist_sellers_dataset.csv")

geolocation = pd.read_csv(r"C:\Users\sanju\OneDrive\Attachments\olist_geolocation_dataset.csv")

category_translation = pd.read_csv(r"C:\Users\sanju\OneDrive\Attachments\product_category_name_translation.csv")

            

In [3]:
print("Customers:", customers.shape)
print("Orders:", orders.shape)
print("Order Items:", order_items.shape)
print("Payments:", payments.shape)
print("Reviews:", reviews.shape)
print("Products:", products.shape)
print("Sellers:", sellers.shape)
print("Geolocation:", geolocation.shape)
print("Category Translation:", category_translation.shape)

Customers: (99441, 5)
Orders: (99441, 8)
Order Items: (112650, 7)
Payments: (103886, 5)
Reviews: (99224, 7)
Products: (32951, 9)
Sellers: (3095, 4)
Geolocation: (1000163, 5)
Category Translation: (71, 2)


In [4]:
orders.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date']

In [5]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors = "coerce")

In [6]:
orders[date_cols].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [7]:
orders["delivery_days"] = (

  orders["order_delivered_customer_date"]
  - orders["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 60 *60)


orders["estimated_delivery_days"] = (
  orders["order_estimated_delivery_date"]
 - orders["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 60 * 60)

orders["delivery_delay_days"] = (
  orders["order_delivered_customer_date"]
  - orders["order_estimated_delivery_date"]
).dt.total_seconds() / (24 * 60 * 60)



In [8]:
orders["delivery_performance"] = np.where(
    orders["order_delivered_customer_date"].isna(),
    "Not Delivered",
  np.where(
orders["delivery_delay_days"] > 0,
"Late",
"On Time / Early"
  )
)

In [9]:
orders[
    [
        "order_id",
        "order_status",
        "delivery_days",
        "estimated_delivery_days",
        "delivery_delay_days",
        "delivery_performance"
    ]
].head()

,order_id,order_status,delivery_days,estimated_delivery_days,delivery_delay_days,delivery_performance
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,8.436574,15.544063,-7.107488,On Time / Early
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,13.782037,19.137766,-5.355729,On Time / Early
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,9.394213,26.639711,-17.245498,On Time / Early
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,13.208750,26.188819,-12.980069,On Time / Early
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,2.873877,12.112049,-9.238171,On Time / Early


In [10]:
orders["delivery_performance"].value_counts()

delivery_performance
On Time / Early    88649
Late                7827
Not Delivered       2965
Name: count, dtype: int64

In [11]:
customers.isnull().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [12]:
customers["customer_id"].duplicated().sum()

np.int64(0)

In [13]:
print("Unique customer id:",
      customers["customer_id"].nunique())

print("Unique customer unique id", customers["customer_id"].nunique())
print("Total customer records:", len(customers))



Unique customer id: 99441
Unique customer unique id 99441
Total customer records: 99441


In [14]:
print("Unique cities:", customers["customer_city"].nunique())
print("Unique states :", customers["customer_state"].nunique())


Unique cities: 4119
Unique states : 27


In [15]:
customers["customer_state"].value_counts().head(10)

customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
Name: count, dtype: int64

In [16]:
# Check whether the same customer has multiple records

customer_record_counts = (
  customers.groupby("customer_unique_id")
  .size()
  .sort_values(ascending = False)
)

customer_record_counts.head(10)

customer_unique_id
8d50f5eadf50201ccdcedfb9e2ac8455    17
3e43e6105506432c953e165fb2acf44c     9
1b6c7548a2a1f9037c1fd3ddfed95f33     7
6469f99c1f9dfae7733b25662e7f1782     7
ca77025e7201e3b30c44b472ff346268     7
47c1a3033b8b77b3ab6e109eb4d5fdf3     6
12f5d6e1cbf93dafd9dcc19095df0b3d     6
63cfc61cee11cbe306bff5857d00bfe4     6
dc813062e0fc23409cd255f7f53c7074     6
de34b16117594161a6a89c50b289d35a     6
dtype: int64

In [17]:
customer_record_counts.value_counts().sort_index()

1     93099
2      2745
3       203
4        30
5         8
6         6
7         3
9         1
17        1
Name: count, dtype: int64

In [18]:
# Order Items Cleaning

# Check duplicate order-item records

print("Duplicated rows:", order_items.duplicated().sum())

print("Duplicate order + item:", order_items.duplicated(
  subset = ["order_id", "order_item_id"]
).sum()
)



Duplicated rows: 0
Duplicate order + item: 0


In [19]:
order_items.isnull().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

In [20]:
order_items[["price", "freight_value"]].describe()

,price,freight_value
count,112650.000000,112650.000000
mean,120.653739,19.990320
std,183.633928,15.806405
min,0.850000,0.000000
25%,39.900000,13.080000
50%,74.990000,16.260000
75%,134.900000,21.150000
max,6735.000000,409.680000


In [21]:
print("Price <= 0:",
      (order_items["price"] <= 0).sum())

print("Freight < 0:",
      (order_items["freight_value"] < 0).sum())



Price <= 0: 0
Freight < 0: 0


In [22]:
# Create item_total

order_items["item_total"] = (
   order_items["price"] + 
   order_items["freight_value"]

)


In [23]:
order_items[
    ["order_id", "price", "freight_value", "item_total"]
].head()

,order_id,price,freight_value,item_total
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,72.19
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,259.83
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,216.87
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,218.04


In [24]:
# Check multiple items per order
items_per_order = (
order_items
.groupby("order_id")
.size()
)
items_per_order

order_id
00010242fe8c5a6d1ba2dd792cb16214    1
00018f77f2f0320c557190d7a144bdd3    1
000229ec398224ef6ca0657da4fc703e    1
00024acbcdf0a6daa1e931b038114c75    1
00042b26cf59d7ce69dfabb4e55b4fd9    1
                                   ..
fffc94f6ce00a00581880bf54a75a037    1
fffcd46ef2263f404302a634eb57f7eb    1
fffce4705a9662cd70adb13d4a31832d    1
fffe18544ffabc95dfada21779c9644f    1
fffe41c64501cc87c801fd61db3f6244    1
Length: 98666, dtype: int64

In [25]:
# eans the freight/shipping cost associated with that order item

In [26]:
# Payments Cleaning

In [27]:
print("Duplicate rows:",
    payments.duplicated().sum()
      )

print("Duplicate order + payment sequence:",
    payments.duplicated(
       subset = ["order_id", "payment_sequential"] 
    )    
      )



Duplicate rows: 0
Duplicate order + payment sequence: 0         False
1         False
2         False
3         False
4         False
          ...  
103881    False
103882    False
103883    False
103884    False
103885    False
Length: 103886, dtype: bool


In [28]:
payments.columns.tolist()

['order_id',
 'payment_sequential',
 'payment_type',
 'payment_installments',
 'payment_value']

In [29]:
payments[[

'order_id',
 'payment_sequential',
 'payment_type',
 'payment_installments',
 'payment_value'
]].head(10)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45
5,298fcdf1f73eb413e4d26d01b25bc1cd,1,credit_card,2,96.12
6,771ee386b001f06208a7419e4fc1bbd7,1,credit_card,1,81.16
7,3d7239c394a212faae122962df514ac7,1,credit_card,3,51.84
8,1f78449c87a54faf9e96e88ba1491fa9,1,credit_card,6,341.09
9,0573b5e23cbd798006520e1d5b4c6714,1,boleto,1,51.95


In [30]:
print("Duplicated order + payment sequence:",
   payments.duplicated(
       subset = ["order_id", "payment_sequential"]
   ).sum()

      )

Duplicated order + payment sequence: 0


In [31]:
payments.isnull().sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

In [32]:
print("payment value <= 0:",
       (payments["payment_value"] <= 0).sum())
   
print("Installments < =0:",
      (payments["payment_installments"] <=0).sum()

      )






payment value <= 0: 9
Installments < =0: 2


In [33]:
payments[[
    "payment_value",
    "payment_installments"
]].describe()

,payment_value,payment_installments
count,103886.000000,103886.000000
mean,154.100380,2.853349
std,217.494064,2.687051
min,0.000000,0.000000
25%,56.790000,1.000000
50%,100.000000,1.000000
75%,171.837500,4.000000
max,13664.080000,24.000000


In [34]:
payments["payment_type"].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [35]:
payments[payments["payment_value"] <= 0]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


In [36]:
payments[payments["payment_installments"] <=0]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [37]:
payments[
   (payments["payment_value"] <= 0) |
   (payments["payment_installments"] <= 0)
][[
    "order_id",
    "payment_type",
    "payment_value",
    "payment_installments"

]]

,order_id,payment_type,payment_value,payment_installments
19922,8bcbe01d44d147f901cd3192671144db,voucher,0.00,1
36822,fa65dad1b0e818e3ccc5cb0e39231352,voucher,0.00,1
43744,6ccb433e00daae1283ccc956189c82ae,voucher,0.00,1
46982,744bade1fcf9ff3f31d860ace076d422,credit_card,58.69,0
51280,4637ca194b6387e2d538dc89b124b0ee,not_defined,0.00,1
57411,00b1cb0320190ca0daa2c88b35206009,not_defined,0.00,1
62674,45ed6e85398a87c253db47c2d9f48216,voucher,0.00,1
77885,fa65dad1b0e818e3ccc5cb0e39231352,voucher,0.00,1
79014,1a57108394169c0b47d8f876acc9ba2d,credit_card,129.94,0
94427,c8c528189310eaa44a745b8d9d26908b,not_defined,0.00,1


In [38]:
payments[payments["payment_value"] <= 0]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


In [39]:
payments[payments["payment_installments"]<= 0]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [40]:
payments[
    (payments["payment_value"] <= 0) |
    (payments["payment_installments"] <= 0)
][[
    "order_id",
    "payment_sequential",
    "payment_type",
    "payment_installments",
    "payment_value"
]]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.00
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.00
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.00
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.00
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.00
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.00
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.00
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.00


In [41]:
# Check whether these orders were delivered

In [42]:
problem_orders = payments[
   (payments["payment_value"]<= 0) |
   (payments["payment_installments"]<= 0)
]["order_id"]


orders[
    orders["order_id"].isin(problem_orders)
]
[[
"order_id",
    "order_status",
    "order_purchase_timestamp"


]]




[['order_id', 'order_status', 'order_purchase_timestamp']]

In [43]:
payments_clean = payments.copy()

In [44]:
payments_clean

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45
...,...,...,...,...,...
103881,0406037ad97740d563a178ecc7a2075c,1,boleto,1,363.31
103882,7b905861d7c825891d6347454ea7863f,1,credit_card,2,96.80
103883,32609bbb3dd69b3c066a6860554a77bf,1,credit_card,1,47.77
103884,b8b61059626efa996a60be9bb9320e10,1,credit_card,5,369.54


In [45]:
# Convert invalid installments to NaN

In [46]:
payments_clean.loc[
    payments_clean["payment_installments"]<= 0,
    "payment_installments"
] = np.nan

In [47]:
# Check payment types

In [48]:
payments_clean["payment_type"].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [49]:
payments_clean.groupby("payment_type")["payment_value"].agg(
    ["count", "sum", "mean"]
).sort_values("sum", ascending = False)

,count,sum,mean
payment_type,,,
credit_card,76795,12542084.19,163.319021
boleto,19784,2869361.27,145.034435
voucher,5775,379436.87,65.703354
debit_card,1529,217989.79,142.570170
not_defined,3,0.00,0.000000


In [50]:
# Reviews Cleaning

In [51]:
reviews

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53
...,...,...,...,...,...,...,...
99219,574ed12dd733e5fa530cfd4bbf39d7c9,2a8c23fee101d4d5662fa670396eb8da,5,NaN,NaN,2018-07-07 00:00:00,2018-07-14 17:18:30
99220,f3897127253a9592a73be9bdfdf4ed7a,22ec9f0669f784db00fa86d035cf8602,5,NaN,NaN,2017-12-09 00:00:00,2017-12-11 20:06:42
99221,b3de70c89b1510c4cd3d0649fd302472,55d4004744368f5571d1f590031933e4,5,NaN,"Excelente mochila, entrega super rápida. Super...",2018-03-22 00:00:00,2018-03-23 09:10:43
99222,1adeb9d84d72fe4e337617733eb85149,7725825d039fc1f0ceb7635e3f7d9206,4,NaN,NaN,2018-07-01 00:00:00,2018-07-02 12:59:13


In [52]:
reviews.isnull().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [53]:
# Check duplicates

print("Duplicated rows:",
    reviews.duplicated().sum()
      )

print("Duplicated review_id:",
     reviews["review_id"].duplicated().sum()
      )




Duplicated rows: 0
Duplicated review_id: 814


In [54]:
reviews["review_score"].value_counts().sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

In [55]:
# Check valid scores

print(
    "Scores below 1:",
    (reviews["review_score"]< 1).sum()
)


print(
    "Scores abobe 5:",
    (reviews["review_score"]> 5).sum()
)

Scores below 1: 0
Scores abobe 5: 0


In [56]:
#Inspect comments

reviews[
    ["review_score",
     "review_comment_title",
     "review_comment_message"]
].head(10)











,review_score,review_comment_title,review_comment_message
0,4,NaN,NaN
1,5,NaN,NaN
2,5,NaN,NaN
3,5,NaN,Recebi bem antes do prazo estipulado.
4,5,NaN,Parabéns lojas lannister adorei comprar pela I...
5,1,NaN,NaN
6,5,NaN,NaN
7,5,NaN,NaN
8,5,NaN,NaN
9,4,recomendo,aparelho eficiente. no site a marca do aparelh...


In [57]:
# Create Review Sentiment Category

reviews["review_sentiment"] = np.where(
    reviews["review_score"]<= 2,
    "Negative",
    np.where(
        reviews["review_score"] == 3,
        "Neutral",
        "Positive"
    )
)

In [58]:
reviews["review_sentiment"]

0        Positive
1        Positive
2        Positive
3        Positive
4        Positive
           ...   
99219    Positive
99220    Positive
99221    Positive
99222    Positive
99223    Negative
Name: review_sentiment, Length: 99224, dtype: str

In [59]:
reviews["review_sentiment"].value_counts()

review_sentiment
Positive    76470
Negative    14575
Neutral      8179
Name: count, dtype: int64

In [60]:
reviews["review_sentiment"].value_counts(normalize=True) * 100

review_sentiment
Positive    77.068048
Negative    14.688987
Neutral      8.242965
Name: proportion, dtype: float64

In [61]:
# Products Cleaning

In [62]:
products

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0
...,...,...,...,...,...,...,...,...,...
32946,a0b7d5a992ccda646f2d34e418fff5a0,moveis_decoracao,45.0,67.0,2.0,12300.0,40.0,40.0,40.0
32947,bf4538d88321d0fd4412a93c974510e6,construcao_ferramentas_iluminacao,41.0,971.0,1.0,1700.0,16.0,19.0,16.0
32948,9a7c6041fa9592d9d9ef6cfe62a71f8c,cama_mesa_banho,50.0,799.0,1.0,1400.0,27.0,7.0,27.0
32949,83808703fc0706a22e264b9d75f04a2e,informatica_acessorios,60.0,156.0,2.0,700.0,31.0,13.0,20.0


In [63]:
products.isnull().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [64]:
# Check duplicate products

print("Duplicate rows:",
     products.duplicated().sum()
      )

print("Duplicate product_id:",
      products["product_id"].duplicated().sum()
      )

Duplicate rows: 0
Duplicate product_id: 0


In [65]:
products.isnull().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [67]:
# Check whether these products actually occur in Order Items

missing_category_products = products.loc[
    products["product_category_name"].isna(),
    "product_id"
]

order_items[
    order_items["product_id"].isin(missing_category_products)
].shape




(1603, 8)

In [68]:
products[
    [
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
].describe()

,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32949.000000,32949.000000,32949.000000,32949.000000
mean,2276.472488,30.815078,16.937661,23.196728
std,4282.038731,16.914458,13.637554,12.079047
min,0.000000,7.000000,2.000000,6.000000
25%,300.000000,18.000000,8.000000,15.000000
50%,700.000000,25.000000,13.000000,20.000000
75%,1900.000000,38.000000,21.000000,30.000000
max,40425.000000,105.000000,105.000000,118.000000


In [71]:
missing_category_products

105      a41e356c76fab66334f36de622ecbd3a
128      d8dee61c2034d6d075997acef1870e9b
145      56139431d72cd51f19eb9f7dae4d1617
154      46b48281eb6d663ced748f324108c733
197      5fb61f482620cb672f5e586bb132eae9
                       ...               
32515    b0a0c5dd78e644373b199380612c350a
32589    10dbe0fbaa2c505123c17fdc34a63c56
32616    bd2ada37b58ae94cc838b9c0569fecd8
32772    fa51e914046aab32764c41356b9d4ea4
32852    c4ceee876c82b8328e9c293fa0e1989b
Name: product_id, Length: 610, dtype: str

In [72]:
products["product_category_name"] = (
     products["product_category_name"].fillna("unknown")

)

In [73]:
products[
    [
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
].describe()

,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32949.000000,32949.000000,32949.000000,32949.000000
mean,2276.472488,30.815078,16.937661,23.196728
std,4282.038731,16.914458,13.637554,12.079047
min,0.000000,7.000000,2.000000,6.000000
25%,300.000000,18.000000,8.000000,15.000000
50%,700.000000,25.000000,13.000000,20.000000
75%,1900.000000,38.000000,21.000000,30.000000
max,40425.000000,105.000000,105.000000,118.000000


In [74]:
print("Missing weight:",
      products["product_weight_g"].isna().sum())

print("Missing length:",
      products["product_length_cm"].isna().sum())

print("Missing height:",
      products["product_height_cm"].isna().sum())

print("Missing width:",
      products["product_width_cm"].isna().sum())


Missing weight: 2
Missing length: 2
Missing height: 2
Missing width: 2


In [75]:
# Fill missing product categories

In [76]:
products["product_category_name"] = (
    products["product_category_name"]
    .fillna("unknown")
)

In [77]:
# Handle missing physical dimensions

dimension_cols = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for col in  dimension_cols:
    products[col] = products[col].fillna(
        products[col].median()
    )


In [78]:
products.isnull().sum()

product_id                      0
product_category_name           0
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                0
product_length_cm               0
product_height_cm               0
product_width_cm                0
dtype: int64

In [79]:
products[
    [
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty"
    ]
].describe()

,product_name_lenght,product_description_lenght,product_photos_qty
count,32341.000000,32341.000000,32341.000000
mean,48.476949,771.495285,2.188986
std,10.245741,635.115225,1.736766
min,5.000000,4.000000,1.000000
25%,42.000000,339.000000,1.000000
50%,51.000000,595.000000,1.000000
75%,57.000000,972.000000,3.000000
max,76.000000,3992.000000,20.000000


In [84]:
text_cols = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty"
]

for col in text_cols:
    products[col] = products[col].fillna(
        products[col].median()
    )

In [85]:
products.isnull().sum()

product_id                    0
product_category_name         0
product_name_lenght           0
product_description_lenght    0
product_photos_qty            0
product_weight_g              0
product_length_cm             0
product_height_cm             0
product_width_cm              0
dtype: int64

In [86]:
# Sellers Cleaning


In [87]:
sellers

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP
...,...,...,...,...
3090,98dddbc4601dd4443ca174359b237166,87111,sarandi,PR
3091,f8201cab383e484733266d1906e2fdfa,88137,palhoca,SC
3092,74871d19219c7d518d0090283e03c137,4650,sao paulo,SP
3093,e603cf3fec55f8697c9059638d6c8eb5,96080,pelotas,RS


In [88]:
sellers.shape

(3095, 4)

In [89]:
sellers.isnull().sum()

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

In [90]:
sellers.columns.tolist()

['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']

In [91]:
print("Duplicate rows:",
      sellers.duplicated().sum())
print("Duplicate seller id:",
      sellers["seller_id"].duplicated().sum())

Duplicate rows: 0
Duplicate seller id: 0


In [92]:
print("Unique sellers:",
      sellers["seller_id"].nunique())

print("Unique states:",
      sellers["seller_state"].nunique())

sellers["seller_state"].value_counts().head(10)

Unique sellers: 3095
Unique states: 23


seller_state
SP    1849
PR     349
MG     244
SC     190
RJ     171
RS     129
GO      40
DF      30
ES      23
BA      19
Name: count, dtype: int64

In [93]:
print(sellers["seller_state"].unique())

<ArrowStringArray>
['SP', 'RJ', 'PE', 'PR', 'GO', 'SC', 'BA', 'DF', 'RS', 'MG', 'RN', 'MT', 'CE',
 'PB', 'AC', 'ES', 'RO', 'PI', 'MS', 'SE', 'MA', 'AM', 'PA']
Length: 23, dtype: str


In [94]:
# Category Translation Cleaning

In [95]:
category_translation

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor
...,...,...
66,flores,flowers
67,artes_e_artesanato,arts_and_craftmanship
68,fraldas_higiene,diapers_and_hygiene
69,fashion_roupa_infanto_juvenil,fashion_childrens_clothes


In [96]:
category_translation.shape

(71, 2)

In [97]:
category_translation.isnull().sum()

product_category_name            0
product_category_name_english    0
dtype: int64

In [98]:
category_translation.columns.tolist()

['product_category_name', 'product_category_name_english']

In [99]:
category_translation[[
    "product_category_name",
    "product_category_name_english"
]].dtypes

product_category_name            str
product_category_name_english    str
dtype: object

In [100]:
# Check duplicates

print("Duplicate rows:",
      category_translation.duplicated().sum())

print("Duplicate category names:",
      category_translation["product_category_name"].duplicated().sum())

Duplicate rows: 0
Duplicate category names: 0


In [101]:
# Check whether English categories are unique

category_translation[
    "product_category_name_english"
].nunique()

71

In [102]:
category_translation.head(10)

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor
5,esporte_lazer,sports_leisure
6,perfumaria,perfumery
7,utilidades_domesticas,housewares
8,telefonia,telephony
9,relogios_presentes,watches_gifts


In [103]:
# Geolocation Cleaning

In [104]:
geolocation

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP
...,...,...,...,...,...
1000158,99950,-28.068639,-52.010705,tapejara,RS
1000159,99900,-27.877125,-52.224882,getulio vargas,RS
1000160,99950,-28.071855,-52.014716,tapejara,RS
1000161,99980,-28.388932,-51.846871,david canabarro,RS


In [105]:
geolocation.head(10)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP
5,1012,-23.547762,-46.635361,são paulo,SP
6,1047,-23.546273,-46.641225,sao paulo,SP
7,1013,-23.546923,-46.634264,sao paulo,SP
8,1029,-23.543769,-46.634278,sao paulo,SP
9,1011,-23.547640,-46.636032,sao paulo,SP


In [106]:
geolocation.shape

(1000163, 5)

In [107]:
geolocation.isnull().sum()

geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

In [108]:
geolocation.columns.tolist()

['geolocation_zip_code_prefix',
 'geolocation_lat',
 'geolocation_lng',
 'geolocation_city',
 'geolocation_state']

In [109]:
geolocation.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  str    
 4   geolocation_state            1000163 non-null  str    
dtypes: float64(2), int64(1), str(2)
memory usage: 50.1 MB


In [110]:
geolocation.duplicated().sum()

np.int64(261831)

In [111]:
geolocation.duplicated(
    subset =[
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng"
    ]
).sum()

np.int64(280009)

In [112]:
# Check coordinates

geolocation[
    ["geolocation_lat","geolocation_lng"]
].describe()

,geolocation_lat,geolocation_lng
count,1.000163e+06,1.000163e+06
mean,-2.117615e+01,-4.639054e+01
std,5.715866e+00,4.269748e+00
min,-3.660537e+01,-1.014668e+02
25%,-2.360355e+01,-4.857317e+01
50%,-2.291938e+01,-4.663788e+01
75%,-1.997962e+01,-4.376771e+01
max,4.506593e+01,1.211054e+02


In [113]:
geolocation["geolocation_state"].value_counts()

geolocation_state
SP    404268
MG    126336
RJ    121169
RS     61851
PR     57859
SC     38328
BA     36045
GO     20139
ES     16748
PE     16432
DF     12986
MT     12031
CE     11674
PA     10853
MS     10431
MA      7853
PB      5538
RN      5041
PI      4549
AL      4183
TO      3576
SE      3563
RO      3478
AM      2432
AC      1301
AP       853
RR       646
Name: count, dtype: int64

In [114]:
geolocation["geolocation_state"].nunique()

27

In [115]:
datasets = {
    "Customers": customers,
    "Orders": orders,
    "Order Items": order_items,
    "Payments": payments_clean,
    "Reviews": reviews,
    "Products": products,
    "Sellers": sellers,
    "Geolocation": geolocation,
    "Category Translation": category_translation
}

for name,df in datasets.items():
    print(f"\n{name}")
    print(df.isnull().sum()[df.isnull().sum() >0])



Customers
Series([], dtype: int64)

Orders
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
delivery_days                    2965
delivery_delay_days              2965
dtype: int64

Order Items
Series([], dtype: int64)

Payments
payment_installments    2
dtype: int64

Reviews
review_comment_title      87656
review_comment_message    58247
dtype: int64

Products
Series([], dtype: int64)

Sellers
Series([], dtype: int64)

Geolocation
Series([], dtype: int64)

Category Translation
Series([], dtype: int64)


In [116]:
for name, df in datasets.items():
    print(
        f"{name}: {df.duplicated().sum()} duplicated rows"
    )

Customers: 0 duplicated rows
Orders: 0 duplicated rows
Order Items: 0 duplicated rows
Payments: 0 duplicated rows
Reviews: 0 duplicated rows
Products: 0 duplicated rows
Sellers: 0 duplicated rows
Geolocation: 261831 duplicated rows
Category Translation: 0 duplicated rows


In [130]:
import os

print(os.getcwd())

print(os.listdir())

import os

print(os.listdir(
    r"C:\Users\sanju\OneDrive\E-Commerce-Customer-Intelligence\data"
))


c:\Users\sanju\OneDrive\E-Commerce-Customer-Intelligence\notebooks
['01_data_exploration.ipynb', '02_data_cleaning.ipynb']
['raw']


In [131]:
import os

os.makedirs("../data/processed", exist_ok=True)

In [132]:
os.listdir("../data")

['processed', 'raw']

In [133]:
customers.to_csv("../data/processed/customers_clean.csv", index=False)

orders.to_csv("../data/processed/orders_clean.csv", index=False)

order_items.to_csv("../data/processed/order_items_clean.csv", index=False)

payments_clean.to_csv("../data/processed/payments_clean.csv", index=False)

reviews.to_csv("../data/processed/reviews_clean.csv", index=False)

products.to_csv("../data/processed/products_clean.csv", index=False)

sellers.to_csv("../data/processed/sellers_clean.csv", index=False)

geolocation.to_csv("../data/processed/geolocation_clean.csv", index=False)

category_translation.to_csv(
    "../data/processed/category_translation_clean.csv",
    index=False
)

In [134]:
os.listdir("../data/processed")

['category_translation_clean.csv',
 'customers_clean.csv',
 'geolocation_clean.csv',
 'orders_clean.csv',
 'order_items_clean.csv',
 'payments_clean.csv',
 'products_clean.csv',
 'reviews_clean.csv',
 'sellers_clean.csv']